## The goal
Build a 2D matrix item_features of shape (n_books, 11) where each row is a single book's feature vector:


[ genre_0, genre_1, ..., genre_9, normalized_pages ]
        10 genre dims              1 page dim
Later you'll concatenate this with the 384-dim description embedding from your .npy to get the final ~395-dim ItemTower input. (Your plan says ~410 because it assumed ~25 genres; you have 10, so it's ~395. No big deal — the tower architecture just adapts.)

The whole step is essentially: turn raw book metadata into clean numeric vectors of fixed shape, in a fixed order, with normalization params saved so inference matches training.

In [10]:
import json 
from mybookrec import DATA_DIR, ROOT_DIR
import polars as pl
import numpy as np

In [11]:
# Load vocab + parquet. The `genres` column is a Polars Struct whose field names
# should be exactly the vocab. Verify with one set comparison instead of a per-row loop.
books_with_genres = pl.read_parquet(DATA_DIR / "transformed" / "books_with_genres.parquet")
with open(ROOT_DIR / "mybookrec" / "features" / "genre_vocab.json", "r") as f:
    genre_vocab = json.load(f)

struct_fields = set(books_with_genres["genres"].struct.fields)
vocab_set = set(genre_vocab)
assert struct_fields == vocab_set, (
    f"Schema mismatch.\n  in struct, not vocab: {struct_fields - vocab_set}\n  in vocab, not struct: {vocab_set - struct_fields}"
)
print(f"Loaded {len(books_with_genres):,} books; {len(genre_vocab)} genres in vocab; schema matches.")

Loaded 1,782,579 books; 10 genres in vocab; schema matches.


In [12]:
# Build (n_books, n_genres) count-weighted genre matrix, then L2 normalize.
# Iterate the vocab (10 vectorized column extractions) instead of looping per-row over 1.78M books.
n_books = books_with_genres.shape[0]
n_genres = len(genre_vocab)

genre_matrix = np.zeros((n_books, n_genres), dtype=np.float32)
for col_idx, genre_name in enumerate(genre_vocab):
    counts = (
        books_with_genres["genres"]
        .struct.field(genre_name)
        .fill_null(0)
        .to_numpy()
    )
    genre_matrix[:, col_idx] = counts

# How many books have no genre data at all? These will be all-zero rows.
zero_rows = int((genre_matrix.sum(axis=1) == 0).sum())
print(f"Books with all-zero genre counts: {zero_rows:,} / {n_books:,}")

# L2 normalize. Guard zero-norm rows so they stay [0, 0, ...] instead of becoming NaN.
norms = np.linalg.norm(genre_matrix, axis=1, keepdims=True)
safe_norms = np.where(norms > 0, norms, 1.0)
genre_matrix = genre_matrix / safe_norms

# Save as .npy — native numpy format, preserves dtype/shape, fast round-trip.
np.save(DATA_DIR / "transformed" / "genre_matrix.npy", genre_matrix)
print(f"Saved genre matrix of shape {genre_matrix.shape} to data/transformed/genre_matrix.npy")

Books with all-zero genre counts: 163,604 / 1,782,579
Saved genre matrix of shape (1782579, 10) to data/transformed/genre_matrix.npy


In [14]:
# Normalize num_pages to [0, 1] using percentile-clipped min-max.
# - Treat 0 and null as "missing" (book metadata where page count was unknown).
# - Impute missing with the median of non-missing values.
# - Clip to [p1, p99] to keep extreme outliers from squashing the scale.
# - Save (p1, p99, median) so inference reuses the same params.
num_pages = books_with_genres["num_pages"].cast(pl.Float64).to_numpy()

missing_mask = np.isnan(num_pages) | (num_pages <= 0)
median = float(np.median(num_pages[~missing_mask]))
num_pages = np.where(missing_mask, median, num_pages)

p1, p99 = np.percentile(num_pages, [1, 99])
p1, p99 = float(p1), float(p99)
normalized_pages = np.clip(num_pages, p1, p99)
normalized_pages = ((normalized_pages - p1) / (p99 - p1)).astype(np.float32)

print(f"num_pages: missing={int(missing_mask.sum()):,}, median={median:.1f}, p1={p1:.1f}, p99={p99:.1f}")
print(f"percentage of pages missing: {missing_mask.mean() * 100:.2f}%")
print(f"normalized range: [{normalized_pages.min():.3f}, {normalized_pages.max():.3f}]")

# Save the normalized vector and the params needed at inference time.
np.save(DATA_DIR / "transformed" / "num_pages_normalized.npy", normalized_pages)
with open(DATA_DIR / "transformed" / "num_pages_norm_params.json", "w") as f:
    json.dump({"p1": p1, "p99": p99, "median": median}, f, indent=2)

num_pages: missing=541,931, median=256.0, p1=14.0, p99=768.0
percentage of pages missing: 30.40%
normalized range: [0.000, 1.000]
